# Training Notebook

**Dual mode:**
- `SAMPLE_SIZE = 500` → CPU local test (< 5 min)
- `SAMPLE_SIZE = -1` → Full Kaggle run (2x T4, uses Accelerate)

Outputs saved to `results/experiments/`.

In [ ]:
import numpy as np
import pandas as pd
import json
import time
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import roc_auc_score

# ── Config ────────────────────────────────────────────────────────
EXP_ID      = 'EXP-001'          # update for each experiment
SAMPLE_SIZE = 500                 # -1 = full data
N_FOLDS     = 5
SEED        = 42
TARGET_COL  = 'target'           # change to actual target
TASK        = 'classification'   # 'classification' or 'regression'

DATA_DIR   = Path('../data/raw')
PROC_DIR   = Path('../data/processed')
RESULT_DIR = Path(f'../results/experiments')
RESULT_DIR.mkdir(parents=True, exist_ok=True)

print(f'EXP_ID = {EXP_ID}')
print(f'SAMPLE_SIZE = {SAMPLE_SIZE}')
print(f'N_FOLDS = {N_FOLDS}')

In [ ]:
# ── Load data ─────────────────────────────────────────────────────
train_path = DATA_DIR / 'train.csv'
test_path  = DATA_DIR / 'test.csv'

if train_path.exists():
    train = pd.read_csv(train_path, nrows=SAMPLE_SIZE if SAMPLE_SIZE > 0 else None)
    test  = pd.read_csv(test_path,  nrows=SAMPLE_SIZE if SAMPLE_SIZE > 0 else None)
else:
    print('WARNING: No data. Using synthetic data.')
    rng = np.random.default_rng(SEED)
    n   = max(SAMPLE_SIZE, 500) if SAMPLE_SIZE > 0 else 1000
    train = pd.DataFrame({
        'id': range(n),
        'feat_1': rng.normal(0, 1, n),
        'feat_2': rng.uniform(0, 10, n),
        'feat_3': rng.choice([0, 1, 2], n),
        TARGET_COL: rng.integers(0, 2, n),
    })
    test = train.drop(columns=[TARGET_COL]).copy()

print(f'Train: {train.shape}  Test: {test.shape}')

In [ ]:
# ── Feature engineering ───────────────────────────────────────────
FEATURE_COLS = [c for c in train.columns if c not in ['id', TARGET_COL]]

# Add engineered features here:
# train['feat_ratio'] = train['feat_1'] / (train['feat_2'] + 1e-6)
# test['feat_ratio']  = test['feat_1']  / (test['feat_2']  + 1e-6)

X      = train[FEATURE_COLS].values
y      = train[TARGET_COL].values
X_test = test[FEATURE_COLS].values

print(f'Features: {FEATURE_COLS}')
print(f'X shape: {X.shape}')

In [ ]:
# ── Model definition ──────────────────────────────────────────────
try:
    import lightgbm as lgb
    USE_LGB = True
except ImportError:
    from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
    USE_LGB = False
    print('LightGBM not found — using sklearn RandomForest')

LGB_PARAMS = {
    'objective':     'binary' if TASK == 'classification' else 'regression',
    'metric':        'auc'    if TASK == 'classification' else 'rmse',
    'num_leaves':    31,
    'learning_rate': 0.05,
    'n_estimators':  200,      # reduce for local speed
    'subsample':     0.8,
    'colsample_bytree': 0.8,
    'seed':          SEED,
    'verbosity':     -1,
    'n_jobs':        -1,
}

print('Model config set.')
print(json.dumps({k: str(v) for k, v in LGB_PARAMS.items()}, indent=2))

In [ ]:
# ── Cross-validation ──────────────────────────────────────────────
if TASK == 'classification':
    kf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    splitter = kf.split(X, y)
else:
    kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    splitter = kf.split(X)

oof_preds  = np.zeros(len(train))
test_preds = np.zeros(len(test))
fold_scores = []

t0 = time.time()

for fold, (train_idx, val_idx) in enumerate(splitter):
    X_tr, X_val = X[train_idx], X[val_idx]
    y_tr, y_val = y[train_idx], y[val_idx]

    if USE_LGB:
        model = lgb.LGBMClassifier(**LGB_PARAMS) if TASK == 'classification' \
                else lgb.LGBMRegressor(**LGB_PARAMS)
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)],
        )
        val_pred  = model.predict_proba(X_val)[:, 1] if TASK == 'classification' \
                    else model.predict(X_val)
        test_pred = model.predict_proba(X_test)[:, 1] if TASK == 'classification' \
                    else model.predict(X_test)
    else:
        clf = RandomForestClassifier(n_estimators=50, random_state=SEED, n_jobs=-1) \
              if TASK == 'classification' \
              else RandomForestRegressor(n_estimators=50, random_state=SEED, n_jobs=-1)
        clf.fit(X_tr, y_tr)
        val_pred  = clf.predict_proba(X_val)[:, 1] if TASK == 'classification' \
                    else clf.predict(X_val)
        test_pred = clf.predict_proba(X_test)[:, 1] if TASK == 'classification' \
                    else clf.predict(X_test)

    oof_preds[val_idx]  = val_pred
    test_preds         += test_pred / N_FOLDS

    if TASK == 'classification':
        score = roc_auc_score(y_val, val_pred)
    else:
        from sklearn.metrics import mean_squared_error
        score = mean_squared_error(y_val, val_pred, squared=False)

    fold_scores.append(score)
    elapsed = time.time() - t0
    print(f'Fold {fold+1}/{N_FOLDS}  score={score:.5f}  elapsed={elapsed:.1f}s')

# ── Final CV score ────────────────────────────────────────────────
if TASK == 'classification':
    cv_score = roc_auc_score(y, oof_preds)
else:
    from sklearn.metrics import mean_squared_error
    cv_score = mean_squared_error(y, oof_preds, squared=False)

print()
print(f'==============================')
print(f'CV Score ({EXP_ID}): {cv_score:.5f}')
print(f'Fold scores: {[round(s, 5) for s in fold_scores]}')
print(f'Std: {np.std(fold_scores):.5f}')
print(f'Total time: {time.time()-t0:.1f}s')
print(f'==============================')

In [ ]:
# ── Save outputs ──────────────────────────────────────────────────
np.save(RESULT_DIR / f'{EXP_ID}_oof.npy',       oof_preds)
np.save(RESULT_DIR / f'{EXP_ID}_test_pred.npy', test_preds)

config = {
    'exp_id':      EXP_ID,
    'cv_score':    round(float(cv_score), 6),
    'fold_scores': [round(float(s), 6) for s in fold_scores],
    'std':         round(float(np.std(fold_scores)), 6),
    'sample_size': SAMPLE_SIZE,
    'n_folds':     N_FOLDS,
    'features':    FEATURE_COLS,
    'model_params': LGB_PARAMS if USE_LGB else 'RandomForest',
}
with open(RESULT_DIR / f'{EXP_ID}_config.json', 'w') as f:
    json.dump(config, f, indent=2, default=str)

print(f'Saved OOF, test preds, and config to {RESULT_DIR}')
print('Next: record results in experiment_queue.md and memory/previous_runs.md')